<a href="https://colab.research.google.com/github/morganhenshaw/NLSTRadiomics/blob/main/create_metadata_table.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install idc-index

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 85.5/85.5 MB 10.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 20.2/20.2 MB 36.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.0/5.0 MB 86.3 MB/s eta 0:00:00
  Attempting uninstall: duckdb
    Found existing installation: duckdb 1.3.2
    Uninstalling duckdb-1.3.2:
      Successfully uninstalled duckdb-1.3.2


In [2]:
my_ProjectID = "nlst-radiomics"

import os
os.environ["GCP_PROJECT_ID"] = my_ProjectID

from google.colab import auth
auth.authenticate_user()

In [3]:
from google.cloud import bigquery

bq_client = bigquery.Client(my_ProjectID)

selection_query = """
SELECT DISTINCT Manufacturer, ManufacturerModelName
FROM bigquery-public-data.idc_current.dicom_all
WHERE collection_id = 'nlst' AND Modality = 'CT'
"""

selection_result = bq_client.query(selection_query)
selection_df = selection_result.result().to_dataframe()

selection_df

,Manufacturer,ManufacturerModelName
0,GE MEDICAL S]STEMS,LightSpeed QX/i
1,GE MEDICAL SYSTEMS,LightSpeed VCT
2,SIEMENS,Emotion 6
3,GE MEDICAL SYSTEMS,Discovery QX/i
4,SIEMENS,Sensation 10
5,GE MEDICAL SYSTEMS,LightSpeed QX/i
6,GE MEDICAL SYSTEMS,HiSpeed QX/i
7,Philips,Mx8000 IDT 16
8,TOSHIBA,Aquilion
9,Philips,Mx8000 IDT


In [4]:
selection_query = """
SELECT
  COUNT(DISTINCT PatientID) AS num_patients,
  COUNT(DISTINCT StudyInstanceUID) AS num_studies,
  COUNT(DISTINCT SeriesInstanceUID) AS num_series,
FROM bigquery-public-data.idc_current.dicom_all
WHERE collection_id = 'nlst'
"""

selection_result = bq_client.query(selection_query)
selection_df = selection_result.result().to_dataframe()

selection_df

,num_patients,num_studies,num_series
0,26410,73574,590572


In [5]:
patient_cohort_query = """
SELECT
  p.pid AS PatientID,
  p.age AS PatientAge_Years,
  p.gender AS PatientSex,
  p.race AS Race,
  p.cigsmok AS CigaretteSmokingStatus,
  ANY_VALUE(c.de_stag) AS LungCancerStage
FROM `bigquery-public-data.idc_v23_clinical.nlst_prsn` AS p
LEFT JOIN `bigquery-public-data.idc_v23_clinical.nlst_canc` AS c
  ON p.pid = c.pid
JOIN (
  SELECT
    PatientID
  FROM `bigquery-public-data.idc_current.dicom_all`
  WHERE
    collection_id = 'nlst'
    AND Modality = 'CT'
  GROUP BY PatientID
) AS d_filter
ON CAST(p.pid AS STRING) = d_filter.PatientID
GROUP BY
  p.pid,
  p.age,
  p.gender,
  p.race,
  p.cigsmok
"""

patient_cohort_result = bq_client.query(patient_cohort_query)
patient_cohort = patient_cohort_result.result().to_dataframe()

print('First 5 rows of the new patient cohort DataFrame:')
display(patient_cohort.head())
print(f'\nShape of the new patient cohort DataFrame: {patient_cohort.shape}')

First 5 rows of the new patient cohort DataFrame:


,PatientID,PatientAge_Years,PatientSex,Race,CigaretteSmokingStatus,LungCancerStage
0,208329,75,2,1,0,None
1,208752,43,1,2,1,None
2,213651,54,2,1,1,None
3,104070,55,2,2,1,None
4,104640,55,2,1,1,None



Shape of the new patient cohort DataFrame: (26254, 6)


In [6]:
from idc_index import IDCClient
c = IDCClient()
c.fetch_index('clinical_index')

In [7]:
# Derive mapping_dict for LungCancerStage from nlst_canc_info's 'de_stag' column
# Ensure nlst_canc_info is defined before use (assuming 'c' is available from a prior cell)
nlst_canc_info = c.clinical_index[c.clinical_index['short_table_name'] == 'nlst_canc'][['column', 'column_label', 'values']]
de_stag_values = nlst_canc_info[nlst_canc_info['column']=='de_stag']['values'].iloc[0]
mapping_dict = {str(item['option_code']): item['option_description'] for item in de_stag_values if item['option_code'] is not None}
print("LungCancerStage mapping dictionary created from idc_index.")

LungCancerStage mapping dictionary created from idc_index.


In [8]:
nlst_prsn_info = c.clinical_index[c.clinical_index['short_table_name'] == 'nlst_prsn'][['column', 'column_label', 'values']]
# Get mapping for PatientSex from nlst_prsn_info
sex_mapping = {item['option_code']: item['option_description'] for item in nlst_prsn_info[nlst_prsn_info['column']=='gender']['values'].iloc[0] if item['option_code'] != '.M'}
patient_cohort['PatientSex_mapped'] = patient_cohort['PatientSex'].astype(str).map(sex_mapping).fillna('Missing')

# Get mapping for Race from nlst_prsn_info
race_mapping = {item['option_code']: item['option_description'] for item in nlst_prsn_info[nlst_prsn_info['column']=='race']['values'].iloc[0] if item['option_code'] != '.M'}
patient_cohort['Race_mapped'] = patient_cohort['Race'].astype(str).map(race_mapping).fillna('Missing')

# Get mapping for CigaretteSmokingStatus from nlst_prsn_info
smoking_mapping = {item['option_code']: item['option_description'] for item in nlst_prsn_info[nlst_prsn_info['column']=='cigsmok']['values'].iloc[0] if item['option_code'] != '.M'}
patient_cohort['CigaretteSmokingStatus_mapped'] = patient_cohort['CigaretteSmokingStatus'].astype(str).map(smoking_mapping).fillna('Missing')

# LungCancerStage mapping (now derived from idc_index)
patient_cohort['LungCancerStage_mapped'] = patient_cohort['LungCancerStage'].astype(str).map(mapping_dict).fillna('Missing')

print("Coded values have been mapped to human-readable descriptions using idc_index.")

Coded values have been mapped to human-readable descriptions using idc_index.


In [9]:
import numpy as np
import pandas as pd
from google.colab import files

# 1. Define Age Groups
bins = [-np.inf, 55, 60, 65, 70, 75, np.inf]
labels = ['<55 years', '55-59 years', '60-64 years', '65-69 years', '70-74 years', '>=75 years']
patient_cohort['AgeGroup'] = pd.cut(patient_cohort['PatientAge_Years'], bins=bins, labels=labels, right=False)

# Function to calculate counts and percentages for a given column
def get_summary_stats(df, column_name, category_label):
    counts = df[column_name].value_counts(dropna=False)
    percentages = (counts / len(df)) * 100
    summary_df = pd.DataFrame({
        'Characteristic': counts.index,
        'Count': counts.values,
        'Percentage': percentages.values
    })
    summary_df['Category'] = category_label
    return summary_df[['Category', 'Characteristic', 'Count', 'Percentage']]

# Get summary statistics for each column
age_summary = get_summary_stats(patient_cohort, 'AgeGroup', 'Age')
sex_summary = get_summary_stats(patient_cohort, 'PatientSex_mapped', 'Sex')
race_summary = get_summary_stats(patient_cohort, 'Race_mapped', 'Race/Ethnicity')
smoking_summary = get_summary_stats(patient_cohort, 'CigaretteSmokingStatus_mapped', 'Cigarette Smoking Status')
stage_summary = get_summary_stats(patient_cohort, 'LungCancerStage_mapped', 'Lung Cancer Stage')

# Combine all summaries into a single DataFrame
table1_summary = pd.concat([
    age_summary,
    sex_summary,
    race_summary,
    smoking_summary,
    stage_summary
])

# Format percentages for display
table1_summary['Percentage'] = table1_summary['Percentage'].map('{:.2f}%'.format)

print('Table 1-like Summary Statistics:')
display(table1_summary)

# Save and download the DataFrame as CSV
csv_filename = 'patient_cohort_summary.csv'
table1_summary.to_csv(csv_filename, index=False)
files.download(csv_filename)
print(f"\nTable 1-like summary downloaded as '{csv_filename}'")

Table 1-like Summary Statistics:


,Category,Characteristic,Count,Percentage
0,Age,55-59 years,11215,42.72%
1,Age,60-64 years,8033,30.60%
2,Age,65-69 years,4691,17.87%
3,Age,70-74 years,2312,8.81%
4,Age,<55 years,2,0.01%
5,Age,>=75 years,1,0.00%
0,Sex,Male,15512,59.08%
1,Sex,Female,10742,40.92%
0,Race/Ethnicity,White,23969,91.30%
1,Race/Ethnicity,Black or African-American,1135,4.32%


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


Table 1-like summary downloaded as 'patient_cohort_summary.csv'


### Explanation of NLST Clinical Tables:

*   **`nlst_canc` (National Lung Screening Trial Cancer Data)**: This table contains detailed information related to diagnosed lung cancers within the NLST cohort. It includes fields such as `lc_topog` (ICD-O-3 Topography of lung cancer), `de_stag` (overall stage of lung cancer), `path_stag` (pathologic stage), `clinical_stag` (clinical stage), `de_grade` (tumor grade), and various T, N, M staging components. It also contains `candx_days` (days to cancer diagnosis) and `dataset_version`.

*   **`nlst_prsn` (National Lung Screening Trial Person Data)**: This table provides patient-level demographic and baseline information for individuals enrolled in the NLST study. Key columns include `pid` (patient ID), `age` (patient age), `gender`, `educat` (education level), `famhx_lu` (family history of lung cancer), `bmi_cat` (BMI category), `race` (race/ethnicity), and `smk_cat` (smoking category). This table serves as a central hub for patient characteristics.

*   **`nlst_screen` (National Lung Screening Trial Screening Data)**: This table focuses on the details of the screening exams performed as part of the NLST. It likely contains information related to the screening rounds, imaging findings, and diagnostic assessments. Columns include `ctdxqual_graininess` (CT image quality: graininess), `ctdxqual_other` (CT image quality: other reasons for limited diagnostic quality), `ct_recon_filter3/4` (CT reconstruction algorithm/filter details), and `dataset_version`. It also has details regarding `screen_num` (screening number), `exam_date` and `ct_dose` (CT radiation dose).

In [10]:
print('--- Contents of idc_v23_clinical.nlst_canc table ---\n')
nlst_canc_info = c.clinical_index[c.clinical_index['short_table_name'] == 'nlst_canc'][['column', 'column_label', 'values']]
display(nlst_canc_info)
print('\n') # Add a newline for better readability

--- Contents of idc_v23_clinical.nlst_canc table ---



,column,column_label,values
7184,dicom_patient_id,idc_provenance_dicom_patient_id,[]
7185,source_batch,idc_provenance_source_batch,"[{'option_code': '0', 'option_description': No..."
7186,pid,pid,[]
7187,lc_topog,ICD-O-3 Topography: ICD-O-3 topography of lung...,"[{'option_code': 'C34.0', 'option_description'..."
7188,topog_source,Source of samples for ICD-O-3 code: Indicates ...,"[{'option_code': '.M', 'option_description': '..."
7189,de_type,ICD-O-3 Morphology (from histology): Lung canc...,"[{'option_code': '.M', 'option_description': '..."
7190,de_grade,Lung Cancer Grade: Lung cancer grade. For ACR...,"[{'option_code': '1', 'option_description': 'G..."
7191,de_stag,"Stage (""Best"": Path if avail., else Clin): Lun...","[{'option_code': '.M', 'option_description': '..."
7192,path_stag,Pathologic Stage: Pathologic stage of lung can...,"[{'option_code': '.M', 'option_description': '..."
7193,clinical_stag,Clinical Stage: Clinical stage of lung cancer ...,"[{'option_code': '.M', 'option_description': '..."


In [11]:
print('--- Contents of idc_v23_clinical.nlst_screen table ---\n')
nlst_screen_info = c.clinical_index[c.clinical_index['short_table_name'] == 'nlst_screen'][['column', 'column_label', 'values']]
display(nlst_screen_info)

--- Contents of idc_v23_clinical.nlst_screen table ---



,column,column_label,values
7287,dicom_patient_id,idc_provenance_dicom_patient_id,[]
7288,source_batch,idc_provenance_source_batch,"[{'option_code': '0', 'option_description': No..."
7289,pid,pid,[]
7290,ctdxqual,Overall diagnostic quality of CT examination,"[{'option_code': '.M', 'option_description': '..."
7291,study_yr,Study Year of Screen,"[{'option_code': '0', 'option_description': 'T..."
7292,techpara_kvp,Technical parameters: kVp: Technical parameter...,"[{'option_code': '.M', 'option_description': '..."
7293,techpara_ma,Technical parameters: mA: Technical parameters...,"[{'option_code': '.M', 'option_description': '..."
7294,techpara_fov,Technical parameters: Display FOV in cm: Techn...,"[{'option_code': '.M', 'option_description': '..."
7295,techpara_effmas,Technical parameters: Effective mAs: Technical...,"[{'option_code': '.M', 'option_description': '..."
7296,ct_recon_filter1,CT reconstruction algorithm / filter: What CT ...,"[{'option_code': '.M', 'option_description': '..."


In [12]:
print('--- Contents of idc_v23_clinical.nlst_prsn table ---\n')
nlst_prsn_info = c.clinical_index[c.clinical_index['short_table_name'] == 'nlst_prsn'][['column', 'column_label', 'values']]
display(nlst_prsn_info)
print('\n') # Add a newline for better readability

--- Contents of idc_v23_clinical.nlst_prsn table ---



,column,column_label,values
7246,dicom_patient_id,idc_provenance_dicom_patient_id,[]
7247,source_batch,idc_provenance_source_batch,"[{'option_code': '0', 'option_description': No..."
7248,race,Race,"[{'option_code': '1', 'option_description': 'W..."
7249,cigsmok,Smoking status at T0: Cigarette smoking status...,"[{'option_code': '0', 'option_description': 'F..."
7250,gender,Gender,"[{'option_code': '1', 'option_description': 'M..."
7251,age,Age at randomization (in years; whole number),[]
7252,loclhil,Cancer in Left Hilum: Was the primary tumor lo...,"[{'option_code': '.N', 'option_description': '..."
7253,locllow,Cancer in Left lower lobe: Was the primary tum...,"[{'option_code': '.N', 'option_description': '..."
7254,loclup,Cancer in Left upper lobe: Was the primary tum...,"[{'option_code': '.N', 'option_description': '..."
7255,locrhil,Cancer in Right Hilum: Was the primary tumor l...,"[{'option_code': '.N', 'option_description': '..."


In [13]:
seg_query = """
SELECT
  StudyInstanceUID,
  SeriesInstanceUID,
  SeriesDescription,
  Manufacturer,
  ManufacturerModelName,
  SoftwareVersions,
  ContentCreatorName
FROM `bigquery-public-data.idc_current.dicom_all`
WHERE
  collection_id = 'nlst'
  AND Modality = 'SEG'
"""

seg_result = bq_client.query(seg_query)
seg_metadata_df = seg_result.result().to_dataframe()

print('First 5 rows of SEG Series Metadata:')
display(seg_metadata_df.head())
print(f'\nShape of the SEG Series Metadata DataFrame: {seg_metadata_df.shape}')

First 5 rows of SEG Series Metadata:


,StudyInstanceUID,SeriesInstanceUID,SeriesDescription,Manufacturer,ManufacturerModelName,SoftwareVersions,ContentCreatorName
0,1.2.840.113654.2.55.30307861664696542811895697...,1.2.276.0.7230010.3.1.3.313263360.31548.170631...,TotalSegmentator(v1.5.6) Segmentation of Series 1,QIICR,https://github.com/QIICR/dcmqi,[7ae0873],"{'Alphabetic': {'FamilyName': 'IDC', 'GivenNam..."
1,1.2.840.113654.2.55.47169270028332479934872699...,1.2.276.0.7230010.3.1.3.313263360.35897.170631...,TotalSegmentator(v1.5.6) Segmentation of Series 1,QIICR,https://github.com/QIICR/dcmqi,[7ae0873],"{'Alphabetic': {'FamilyName': 'IDC', 'GivenNam..."
2,1.2.840.113654.2.55.73932085069052180760272253...,1.2.276.0.7230010.3.1.3.313263360.8215.1706317...,TotalSegmentator(v1.5.6) Segmentation of Serie...,QIICR,https://github.com/QIICR/dcmqi,[7ae0873],"{'Alphabetic': {'FamilyName': 'IDC', 'GivenNam..."
3,1.2.840.113654.2.55.26451943921554107058782129...,1.2.276.0.7230010.3.1.3.313263360.42983.170633...,TotalSegmentator(v1.5.6) Segmentation of Series 2,QIICR,https://github.com/QIICR/dcmqi,[7ae0873],"{'Alphabetic': {'FamilyName': 'IDC', 'GivenNam..."
4,1.2.840.113654.2.55.26892366813201412158810706...,1.2.276.0.7230010.3.1.3.313263360.44539.170633...,TotalSegmentator(v1.5.6) Segmentation of Series 2,QIICR,https://github.com/QIICR/dcmqi,[7ae0873],"{'Alphabetic': {'FamilyName': 'IDC', 'GivenNam..."



Shape of the SEG Series Metadata DataFrame: (128835, 7)


In [14]:
print(f"\n--- Unique Series Descriptions ({seg_metadata_df['SeriesDescription'].nunique()}) ---")
display(seg_metadata_df['SeriesDescription'].value_counts().head(20))



--- Unique Series Descriptions (5492) ---


,count
SeriesDescription,
TotalSegmentator(v1.5.6) Segmentation of Series 2,47987
TotalSegmentator(v1.5.6) Segmentation of Series 3,38198
TotalSegmentator(v1.5.6) Segmentation of Series 4,13362
TotalSegmentator(v1.5.6) Segmentation of Series 5,6180
TotalSegmentator(v1.5.6) Segmentation of Series 6,5377
TotalSegmentator(v1.5.6) Segmentation of Series 1,4449
TotalSegmentator(v1.5.6) Segmentation of Series 0,1132
AIMI lung and nodule AI segmentation,1042
3d_fullres-tta_nnU-Net_Segmentation,1039


# Task
Filter the `patient_cohort` DataFrame to create a `malignant_patient_cohort` containing only patients with confirmed malignant nodules (i.e., excluding 'Missing', 'TNM not available', 'Occult Carcinoma', 'Carcinoid, cannot be assessed', and 'Unknown/ cannot be assessed' `LungCancerStage_mapped` values). Then, display the first few rows and shape of this new malignant cohort. Finally, generate and display a Table 1-like summary of the `malignant_patient_cohort` including counts and percentages for age groups, sex, race/ethnicity, cigarette smoking status, and lung cancer stage. Summarize the process and key characteristics of the malignant cohort.

## Filter Patient Cohort for Malignancy

### Subtask:
Filter the existing `patient_cohort` DataFrame to include only patients with confirmed malignant nodules. This will be done by excluding patients where `LungCancerStage_mapped` is 'Missing' or represents an unknown/non-malignant status. This implies that the patient had a positive CT scan that led to a cancer diagnosis reflected in the `nlst_canc` table.


**Reasoning**:
First, I will define the `non_malignant_statuses` list as specified in the instructions. Then, I will filter the `patient_cohort` DataFrame to create `malignant_patient_cohort` by excluding patients with non-malignant or unknown lung cancer stages. Finally, I will display the first 5 rows and the shape of the new DataFrame to verify the filtering.



In [15]:
non_malignant_statuses = [
    'Missing',
    'TNM not available',
    'Occult Carcinoma',
    'Carcinoid, cannot be assessed',
    'Unknown, cannot be assessed'
]

malignant_patient_cohort = patient_cohort[
    ~patient_cohort['LungCancerStage_mapped'].isin(non_malignant_statuses)
]

print('First 5 rows of the malignant_patient_cohort DataFrame:')
display(malignant_patient_cohort.head())
print(f'\nShape of the malignant_patient_cohort DataFrame: {malignant_patient_cohort.shape}')

First 5 rows of the malignant_patient_cohort DataFrame:


,PatientID,PatientAge_Years,PatientSex,Race,CigaretteSmokingStatus,LungCancerStage,PatientSex_mapped,Race_mapped,CigaretteSmokingStatus_mapped,LungCancerStage_mapped,AgeGroup
152,213567,55,2,1,1,400,Female,White,Current,Stage IV,55-59 years
219,207286,55,1,1,1,220,Male,White,Current,Stage IIB,55-59 years
289,204120,55,1,1,0,120,Male,White,Former,Stage IB,55-59 years
510,200931,55,1,1,1,400,Male,White,Current,Stage IV,55-59 years
536,107573,55,1,1,1,400,Male,White,Current,Stage IV,55-59 years



Shape of the malignant_patient_cohort DataFrame: (1044, 11)


**Reasoning**:
Now that the `malignant_patient_cohort` DataFrame has been created, I will generate a Table 1-like summary of its characteristics, including age groups, sex, race/ethnicity, cigarette smoking status, and lung cancer stage. This involves reusing the `get_summary_stats` function and concatenating the results for display, similar to how the initial `patient_cohort` summary was generated.



In [16]:
import numpy as np
import pandas as pd

# Reuse the get_summary_stats function (already defined in d024519f)
def get_summary_stats(df, column_name, category_label):
    counts = df[column_name].value_counts(dropna=False)
    percentages = (counts / len(df)) * 100
    summary_df = pd.DataFrame({
        'Characteristic': counts.index,
        'Count': counts.values,
        'Percentage': percentages.values
    })
    summary_df['Category'] = category_label
    return summary_df[['Category', 'Characteristic', 'Count', 'Percentage']]

# Generate AgeGroup for malignant_patient_cohort (if not already present, though it should be inherited)
# Ensure 'AgeGroup' is available, it should have been carried over from patient_cohort. If not, re-create it.
if 'AgeGroup' not in malignant_patient_cohort.columns:
    bins = [-np.inf, 55, 60, 65, 70, 75, np.inf]
    labels = ['<55 years', '55-59 years', '60-64 years', '65-69 years', '70-74 years', '>=75 years']
    malignant_patient_cohort['AgeGroup'] = pd.cut(malignant_patient_cohort['PatientAge_Years'], bins=bins, labels=labels, right=False)

# Get summary statistics for each column for the malignant_patient_cohort
malignant_age_summary = get_summary_stats(malignant_patient_cohort, 'AgeGroup', 'Age')
malignant_sex_summary = get_summary_stats(malignant_patient_cohort, 'PatientSex_mapped', 'Sex')
malignant_race_summary = get_summary_stats(malignant_patient_cohort, 'Race_mapped', 'Race/Ethnicity')
malignant_smoking_summary = get_summary_stats(malignant_patient_cohort, 'CigaretteSmokingStatus_mapped', 'Cigarette Smoking Status')
malignant_stage_summary = get_summary_stats(malignant_patient_cohort, 'LungCancerStage_mapped', 'Lung Cancer Stage')

# Combine all summaries into a single DataFrame for the malignant cohort
malignant_table1_summary = pd.concat([
    malignant_age_summary,
    malignant_sex_summary,
    malignant_race_summary,
    malignant_smoking_summary,
    malignant_stage_summary
])

# Format percentages for display
malignant_table1_summary['Percentage'] = malignant_table1_summary['Percentage'].map('{:.2f}%'.format)

print('Table 1-like Summary Statistics for Malignant Cohort:')
display(malignant_table1_summary)


Table 1-like Summary Statistics for Malignant Cohort:


,Category,Characteristic,Count,Percentage
0,Age,60-64 years,310,29.69%
1,Age,65-69 years,279,26.72%
2,Age,55-59 years,275,26.34%
3,Age,70-74 years,180,17.24%
4,Age,<55 years,0,0.00%
5,Age,>=75 years,0,0.00%
0,Sex,Male,633,60.63%
1,Sex,Female,411,39.37%
0,Race/Ethnicity,White,956,91.57%
1,Race/Ethnicity,Black or African-American,46,4.41%


In [17]:
can_scr_query = """
SELECT
  p.pid AS PatientID,
  p.can_scr AS CancerScreenResult
FROM `bigquery-public-data.idc_v23_clinical.nlst_prsn` AS p
WHERE p.pid IN (
    SELECT DISTINCT pid FROM `bigquery-public-data.idc_v23_clinical.nlst_prsn`
)
"""

can_scr_result = bq_client.query(can_scr_query)
can_scr_df = can_scr_result.result().to_dataframe()

# Ensure PatientID in can_scr_df is of the same type as in patient_cohort for merging
can_scr_df['PatientID'] = can_scr_df['PatientID'].astype(patient_cohort['PatientID'].dtype)

# Merge can_scr_df with patient_cohort
patient_cohort = patient_cohort.merge(can_scr_df, on='PatientID', how='left')

print('First 5 rows of patient_cohort with CancerScreenResult:')
display(patient_cohort.head())
print(f'\nShape of patient_cohort: {patient_cohort.shape}')

First 5 rows of patient_cohort with CancerScreenResult:


,PatientID,PatientAge_Years,PatientSex,Race,CigaretteSmokingStatus,LungCancerStage,PatientSex_mapped,Race_mapped,CigaretteSmokingStatus_mapped,LungCancerStage_mapped,AgeGroup,CancerScreenResult
0,208329,75,2,1,0,None,Female,White,Former,Missing,>=75 years,0
1,208752,43,1,2,1,None,Male,Black or African-American,Current,Missing,<55 years,0
2,213651,54,2,1,1,None,Female,White,Current,Missing,<55 years,0
3,104070,55,2,2,1,None,Female,Black or African-American,Current,Missing,55-59 years,0
4,104640,55,2,1,1,None,Female,White,Current,Missing,55-59 years,0



Shape of patient_cohort: (26254, 12)


In [18]:
# Get mapping for can_scr from nlst_prsn_info
# nlst_prsn_info should be available from previous cells
can_scr_mapping = {item['option_code']: item['option_description'] for item in nlst_prsn_info[nlst_prsn_info['column']=='can_scr']['values'].iloc[0] if item['option_code'] is not None}

patient_cohort['CancerScreenResult_mapped'] = patient_cohort['CancerScreenResult'].astype(str).map(can_scr_mapping).fillna('Missing')

print("CancerScreenResult coded values have been mapped.")

CancerScreenResult coded values have been mapped.


In [19]:
non_malignant_statuses = [
    'Missing',
    'TNM not available',
    'Occult Carcinoma',
    'Carcinoid, cannot be assessed',
    'Unknown, cannot be assessed'
]

malignant_patient_cohort = patient_cohort[
    ~patient_cohort['LungCancerStage_mapped'].isin(non_malignant_statuses)
]

print('First 5 rows of the malignant_patient_cohort DataFrame (with CancerScreenResult):')
display(malignant_patient_cohort.head())
print(f'\nShape of the malignant_patient_cohort DataFrame: {malignant_patient_cohort.shape}')

First 5 rows of the malignant_patient_cohort DataFrame (with CancerScreenResult):


,PatientID,PatientAge_Years,PatientSex,Race,CigaretteSmokingStatus,LungCancerStage,PatientSex_mapped,Race_mapped,CigaretteSmokingStatus_mapped,LungCancerStage_mapped,AgeGroup,CancerScreenResult,CancerScreenResult_mapped
152,213567,55,2,1,1,400,Female,White,Current,Stage IV,55-59 years,4,Post Screening
219,207286,55,1,1,1,220,Male,White,Current,Stage IIB,55-59 years,1,Positive Screen
289,204120,55,1,1,0,120,Male,White,Former,Stage IB,55-59 years,4,Post Screening
510,200931,55,1,1,1,400,Male,White,Current,Stage IV,55-59 years,1,Positive Screen
536,107573,55,1,1,1,400,Male,White,Current,Stage IV,55-59 years,4,Post Screening



Shape of the malignant_patient_cohort DataFrame: (1044, 13)


In [20]:
positive_screen_malignant = malignant_patient_cohort[
    malignant_patient_cohort['CancerScreenResult_mapped'] == 'Positive Screen'
]

count_positive_screen = len(positive_screen_malignant)
percentage_positive_screen = (count_positive_screen / len(malignant_patient_cohort)) * 100

print(f"Total malignant patients: {len(malignant_patient_cohort)}")
print(f"Patients with 'Positive Screen': {count_positive_screen}")
print(f"Percentage of malignant patients with 'Positive Screen': {percentage_positive_screen:.2f}%\n")

print("Value counts for CancerScreenResult_mapped in malignant_patient_cohort:")
display(malignant_patient_cohort['CancerScreenResult_mapped'].value_counts(dropna=False))

Total malignant patients: 1044
Patients with 'Positive Screen': 627
Percentage of malignant patients with 'Positive Screen': 60.06%

Value counts for CancerScreenResult_mapped in malignant_patient_cohort:


,count
CancerScreenResult_mapped,
Positive Screen,627
Post Screening,359
Negative Screen,42
Missed Screen,16


### Finding all CT Series for Patient 100012

In [55]:
patient_id = '100012'

print(f"patient_id: {patient_id}")

patient_id: 100012


In [56]:
import pandas as pd

find_all_ct_series_query = f"""
SELECT
  DISTINCT SeriesInstanceUID, SeriesDescription
FROM `bigquery-public-data.idc_current.dicom_all`
WHERE
  collection_id = 'nlst'
  AND PatientID = '{patient_id}'
  AND Modality = 'CT'
ORDER BY SeriesInstanceUID
"""

find_all_ct_series_result = bq_client.query(find_all_ct_series_query)
find_all_ct_series_df = find_all_ct_series_result.result().to_dataframe()

# Set pandas display option to show full column width
pd.set_option('display.max_colwidth', None)

print(f"All CT Series found for PatientID {patient_id} (full SeriesInstanceUIDs):")
display(find_all_ct_series_df)
print(f'\nTotal CT Series found: {find_all_ct_series_df.shape[0]}')

All CT Series found for PatientID 100012 (full SeriesInstanceUIDs):


,SeriesInstanceUID,SeriesDescription
0,1.2.840.113654.2.55.135088253786049275791463451273034430925,"1,OPA,SE,VZOOM,B30f,280,2,120,70,40,na"
1,1.2.840.113654.2.55.240231128564881525363489796879328810792,"0,OPA,SE,VZOOM,B50f,300,2,120,75,40,na"
2,1.2.840.113654.2.55.335938848402215862539398120263494500079,"0,OPA,SE,VZOOM,B30f,300,2,120,75,40,na"
3,1.2.840.113654.2.55.4303894980888172655039251025765147023,"0,DSL,SE,VZOOM,T20s,512,na,120,144.8,144,na"
4,1.2.840.113654.2.55.50761756412482430061802871163319122196,"1,OPA,SE,VZOOM,B50f,280,2,120,70,40,na"



Total CT Series found: 5


In [62]:
ct_series_instance_uid = '1.2.840.113654.2.55.50761756412482430061802871163319122196'

seg_query_nested = f"""
SELECT
  t.SeriesInstanceUID,
  t.SeriesDescription,
  seg.SegmentLabel,
  t.SOPInstanceUID
FROM `bigquery-public-data.idc_current.dicom_all` AS t,
  UNNEST(t.ReferencedSeriesSequence) AS ref_series,
  UNNEST(t.SegmentSequence) AS seg
WHERE
  t.collection_id = 'nlst'
  AND t.PatientID = '{patient_id}'
  AND t.Modality = 'SEG'
  AND ref_series.SeriesInstanceUID = '{ct_series_instance_uid}'
  AND seg.SegmentLabel IN ('Lung', 'Nodule')
"""

seg_result_nested = bq_client.query(seg_query_nested)
seg_df_nested = seg_result_nested.result().to_dataframe()

print('Identified DICOM SEG files with nested SegmentLabel:')
display(seg_df_nested)
print(f'\nShape of the filtered SEG DataFrame using UNNEST: {seg_df_nested.shape}')

Identified DICOM SEG files with nested SegmentLabel:


,SeriesInstanceUID,SeriesDescription,SegmentLabel,SOPInstanceUID
0,1.2.276.0.7230010.3.1.3.17436516.3063738.1720665129.785873,AIMI lung and nodule AI segmentation,Lung,1.2.276.0.7230010.3.1.4.17436516.3063738.1720665129.785874
1,1.2.276.0.7230010.3.1.3.17436516.3063738.1720665129.785873,AIMI lung and nodule AI segmentation,Nodule,1.2.276.0.7230010.3.1.4.17436516.3063738.1720665129.785874



Shape of the filtered SEG DataFrame using UNNEST: (2, 4)
